In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
# ============================================================
# Cell 1 — Imports & Setup
# ============================================================

import glob
import random
import warnings
import joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import optuna
import xgboost as xgb

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.utils.class_weight import compute_sample_weight

from sklearn.metrics import (
    classification_report,
    balanced_accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_recall_fscore_support,
)

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

# ------------------------------------------------------------
# Dataset paths
# ------------------------------------------------------------
FEATURES_PATH = "/kaggle/input/datasets/amarnathdj/classifier-features"
OUTPUT_DIR = "/kaggle/working"

# ------------------------------------------------------------
# Time-series Cross Validation
# ------------------------------------------------------------
N_SPLITS = 5
tscv = TimeSeriesSplit(n_splits=N_SPLITS)

print("Environment ready.")

In [ ]:
# ============================================================
# Cell 2 — Load Monthly Feature Files
# ============================================================

files = sorted(glob.glob(f"{FEATURES_PATH}/features_*.parquet"))
files = [f for f in files if "combined" not in f]

print(f"Found {len(files)} monthly feature files.\n")

dfs = []

for file in files:

    tmp = pd.read_parquet(file)

    # Ensure timestamp is datetime
    tmp["timestamp"] = pd.to_datetime(tmp["timestamp"])

    month = file.split("features_")[-1].replace(".parquet", "")
    tmp["source_month"] = month

    dfs.append(tmp)

    print(f"{month:<30} {tmp.shape}")

# ------------------------------------------------------------
# Combine all months
# ------------------------------------------------------------

df = pd.concat(dfs, ignore_index=True)

# Chronological ordering
df = (
    df
    .sort_values("timestamp")
    .reset_index(drop=True)
)

print("\n==============================")
print("Dataset Summary")
print("==============================")

print(f"Shape            : {df.shape}")
print(f"Date range       : {df.timestamp.min()}  -->  {df.timestamp.max()}")
print(f"Source months    : {df.source_month.nunique()}")
print(f"Duplicate rows   : {df.duplicated().sum()}")
print(f"Missing values   : {df.isna().sum().sum()}")

assert df["timestamp"].is_monotonic_increasing, \
    "Dataset is NOT sorted chronologically!"

print("\nDataset loaded successfully.")

In [ ]:
# ============================================================
# Cell 2A — Dataset Overview
# ============================================================

fig, ax = plt.subplots(1, 2, figsize=(16,5))

# ------------------------------------
# Samples per month
# ------------------------------------

(
    df["source_month"]
    .value_counts()
    .sort_index()
    .plot(
        kind="bar",
        ax=ax[0]
    )
)

ax[0].set_title("Samples per Month")
ax[0].set_xlabel("")
ax[0].set_ylabel("Number of Samples")
ax[0].tick_params(axis="x", rotation=90)

# ------------------------------------
# Rainfall distribution
# ------------------------------------

ax[1].hist(
    df["rainfall_mm"],
    bins=50,
    edgecolor="black"
)

ax[1].set_title("Rainfall Distribution")
ax[1].set_xlabel("Rainfall (mm)")
ax[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Cell 2B — Hourly Aggregation (WMO-aligned)
# ============================================================
# Group all clips/intervals into 1-hour windows.
# rainfall_mm is SUMMED across the hour  → gives mm/hr directly.
# All audio features are AVERAGED across clips in the hour.
# This aligns labels with WMO thresholds (set in Cell 3).
# ============================================================

df["hour"] = df["timestamp"].dt.floor("h")

# Identify feature columns now (everything except known meta)
_KNOWN_META = {
    "timestamp", "rainfall_mm", "wav_count",
    "rain", "n_clips_used", "0_mean", "0_std",
    "source_month", "hour",
}
_feat_cols_tmp = [c for c in df.columns if c not in _KNOWN_META]

# Build aggregation dict
_agg = {c: "mean" for c in _feat_cols_tmp}
_agg["rainfall_mm"] = "sum"   # sum across hour = mm / hr

df_hourly = (
    df
    .groupby("hour")
    .agg(_agg)
    .reset_index()
    .rename(columns={"hour": "timestamp"})
    .sort_values("timestamp")
    .reset_index(drop=True)
)

df_hourly["source_month"] = (
    df_hourly["timestamp"].dt.to_period("M").astype(str)
)

df = df_hourly

print("==============================")
print("Hourly Dataset Summary")
print("==============================")
print(f"Hourly samples   : {len(df)}")
print(f"Date range       : {df.timestamp.min()}  -->  {df.timestamp.max()}")
print(f"Rainfall col     : rainfall_mm now represents mm/hr (hourly sum)")
print(f"Missing values   : {df.isna().sum().sum()}")


In [ ]:
# ============================================================
# Cell 3 — Create 4-Class Labels (WMO Standard)
# ============================================================

# ------------------------------------------------------------
# Thresholds  (mm / hr)
# ------------------------------------------------------------
# Class 0 : No Rain       — 0 mm/hr
# Class 1 : Light Rain    — 0   < x < 2.5
# Class 2 : Moderate Rain — 2.5 <= x < 7.6
# Class 3 : Heavy Rain    — >= 7.6
#
# Modify ONLY these three values for experiments.
# ------------------------------------------------------------

SILENT_THRESH   = 0.0    # No Rain boundary
LIGHT_THRESH    = 2.5    # Light / Moderate boundary  (WMO)
MODERATE_THRESH = 7.6    # Moderate / Heavy boundary  (WMO)

assert MODERATE_THRESH > LIGHT_THRESH > SILENT_THRESH, \
    "Thresholds must satisfy: 0 < LIGHT_THRESH < MODERATE_THRESH"

# ------------------------------------------------------------
# Label generation
# ------------------------------------------------------------

def make_label(mm_hr):
    """WMO-standard 4-class labeller (units: mm/hr)."""
    if mm_hr <= SILENT_THRESH:
        return 0           # No Rain
    elif mm_hr < LIGHT_THRESH:
        return 1           # Light Rain     (0  – 2.5)
    elif mm_hr < MODERATE_THRESH:
        return 2           # Moderate Rain  (2.5 – 7.6)
    else:
        return 3           # Heavy Rain     (>= 7.6)


df["label"] = df["rainfall_mm"].apply(make_label)

CLASS_NAMES = {
    0: "No Rain",
    1: "Light Rain",
    2: "Moderate Rain",
    3: "Heavy Rain",
}

NUM_CLASSES = len(CLASS_NAMES)

counts = (
    df["label"]
    .value_counts()
    .sort_index()
)

print("==============================")
print("Class Distribution")
print("==============================")

for cls, count in counts.items():
    print(
        f"{CLASS_NAMES[cls]:14s}"
        f"{count:8d}"
        f" ({100*count/len(df):5.2f}%)"
    )

print()
print(f"Total samples : {len(df)}")


In [ ]:
# ============================================================
# Cell 3A — Class Distribution
# ============================================================

plt.figure(figsize=(7, 5))

counts.plot(
    kind="bar",
    edgecolor="black"
)

plt.xticks(
    range(NUM_CLASSES),
    list(CLASS_NAMES.values()),
    rotation=0
)

plt.ylabel("Number of Samples")
plt.title("Class Distribution (WMO 4-Class)")

for i, v in enumerate(counts.values):
    plt.text(
        i,
        v,
        str(v),
        ha="center",
        va="bottom",
        fontsize=10
    )

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# Cell 4 — Feature Selection & Dataset Preparation
# ============================================================

# Columns that are NOT model features.
# After hourly aggregation, dropped columns (wav_count, rain, n_clips_used,
# 0_mean, 0_std) no longer exist in df — no need to guard against them.
META_COLS = [
    "timestamp",
    "rainfall_mm",
    "source_month",
    "label",
]

# Feature columns — everything in df that is not a meta column
FEATURE_COLS = [c for c in df.columns if c not in META_COLS]

print(f"Number of feature columns : {len(FEATURE_COLS)}")

# ------------------------------------------------------------
# Remove rows containing missing feature values
# ------------------------------------------------------------

before_rows = len(df)

df = (
    df
    .dropna(subset=FEATURE_COLS)
    .reset_index(drop=True)
)

after_rows = len(df)

print(f"Rows before cleaning : {before_rows}")
print(f"Rows after cleaning  : {after_rows}")
print(f"Rows removed         : {before_rows-after_rows}")

# ------------------------------------------------------------
# Create feature matrix and target
# ------------------------------------------------------------

X = df[FEATURE_COLS].to_numpy(dtype=np.float32)
y = df["label"].to_numpy(dtype=np.int32)

print("\nDataset ready for modelling")

print(f"X shape : {X.shape}")
print(f"y shape : {y.shape}")

print("\nClass counts:")

for cls, count in zip(*np.unique(y, return_counts=True)):
    print(f"{CLASS_NAMES[cls]:12s}: {count}")

In [ ]:
# ============================================================
# Cell 5 — Feature Scaling & Sample Weights
# ============================================================

# ------------------------------------------------------------
# Scale Features
# ------------------------------------------------------------

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

joblib.dump(
    scaler,
    f"{OUTPUT_DIR}/scaler_3class.pkl"
)

print("Scaler saved.")

# ------------------------------------------------------------
# Compute balanced sample weights
# ------------------------------------------------------------

sample_weights = compute_sample_weight(
    class_weight="balanced",
    y=y,
)

print("\nSample Weights")

for cls in sorted(np.unique(y)):

    weight = sample_weights[y == cls].mean()

    print(
        f"{CLASS_NAMES[cls]:12s}"
        f"{weight:.3f}"
    )

In [ ]:
# ============================================================
# Cell 6 — Optuna Hyperparameter Search
# ============================================================

def objective(trial):

    params = {

        "objective": "multi:softprob",
        "num_class": NUM_CLASSES,
        "eval_metric": "mlogloss",

        "n_estimators": trial.suggest_int(
            "n_estimators",
            200,
            1000
        ),

        "max_depth": trial.suggest_int(
            "max_depth",
            3,
            10
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.005,
            0.30,
            log=True
        ),

        "subsample": trial.suggest_float(
            "subsample",
            0.60,
            1.00
        ),

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree",
            0.50,
            1.00
        ),

        "min_child_weight": trial.suggest_int(
            "min_child_weight",
            1,
            20
        ),

        "gamma": trial.suggest_float(
            "gamma",
            0,
            5
        ),

        "reg_alpha": trial.suggest_float(
            "reg_alpha",
            0,
            5
        ),

        "reg_lambda": trial.suggest_float(
            "reg_lambda",
            0.1,
            10
        ),

        "random_state": SEED,
        "n_jobs": -1,
        "verbosity": 0,

    }

    fold_scores = []

    for train_idx, val_idx in tscv.split(X_scaled):

        X_train = X_scaled[train_idx]
        X_val = X_scaled[val_idx]

        y_train = y[train_idx]
        y_val = y[val_idx]

        sw_train = sample_weights[train_idx]

        model = xgb.XGBClassifier(**params)

        model.fit(

            X_train,
            y_train,

            sample_weight=sw_train,

            verbose=False

        )

        pred = model.predict(X_val)

        score = f1_score(

            y_val,
            pred,

            average="macro",

            zero_division=0

        )

        fold_scores.append(score)

    return np.mean(fold_scores)


print("=" * 60)
print("Running Optuna Hyperparameter Search")
print("=" * 60)

study = optuna.create_study(

    direction="maximize",

    sampler=optuna.samplers.TPESampler(

        seed=SEED

    )

)

study.optimize(

    objective,

    n_trials=80,

    show_progress_bar=True

)

best_params = study.best_params

best_params.update({

    "objective": "multi:softprob",
    "num_class": NUM_CLASSES,
    "eval_metric": "mlogloss",

    "random_state": SEED,

    "n_jobs": -1,

    "verbosity": 1,

})

print("\nBest CV Macro-F1 :", round(study.best_value,4))

print("\nBest Parameters\n")

for k,v in best_params.items():

    print(f"{k:25s}: {v}")

In [ ]:
# ============================================================
# Cell 7 — Cross-Validation Evaluation
# ============================================================

print("=" * 60)
print("Cross Validation Evaluation")
print("=" * 60)

all_preds = np.full(len(y), -1, dtype=int)
cv_mask = np.zeros(len(y), dtype=bool)

fold_scores = []

for fold, (train_idx, val_idx) in enumerate(tscv.split(X_scaled), start=1):

    X_train = X_scaled[train_idx]
    X_val = X_scaled[val_idx]

    y_train = y[train_idx]
    y_val = y[val_idx]

    sw_train = sample_weights[train_idx]

    model = xgb.XGBClassifier(**best_params)

    model.fit(
        X_train,
        y_train,
        sample_weight=sw_train,
        verbose=False,
    )

    preds = model.predict(X_val)

    all_preds[val_idx] = preds
    cv_mask[val_idx] = True

    macro_f1 = f1_score(
        y_val,
        preds,
        average="macro",
        zero_division=0,
    )

    fold_scores.append(macro_f1)

    print(
        f"Fold {fold} | "
        f"Macro-F1 = {macro_f1:.4f} | "
        f"Validation Samples = {len(val_idx)} | "
        f"Rain = {(y_val>0).mean()*100:.2f}%"
    )

# ------------------------------------------------------------
# Final Report
# ------------------------------------------------------------

y_true = y[cv_mask]
y_pred = all_preds[cv_mask]

print("\n")
print("=" * 60)
print("FINAL CROSS VALIDATION REPORT")
print("=" * 60)

print(classification_report(
    y_true,
    y_pred,
    target_names=list(CLASS_NAMES.values()),
    zero_division=0,
))

bal_acc = balanced_accuracy_score(y_true, y_pred)
macro_f1 = f1_score(
    y_true,
    y_pred,
    average="macro",
    zero_division=0,
)

print(f"Balanced Accuracy : {bal_acc:.4f}")
print(f"Macro F1          : {macro_f1:.4f}")

# ------------------------------------------------------------
# Confusion Matrix
# ------------------------------------------------------------

cm = confusion_matrix(y_true, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=list(CLASS_NAMES.values()),
)

fig, ax = plt.subplots(figsize=(7,7))

disp.plot(
    cmap="Blues",
    ax=ax,
    colorbar=False,
)

plt.title("Cross-Validation Confusion Matrix")

plt.tight_layout()

plt.savefig(
    f"{OUTPUT_DIR}/confusion_matrix_cv.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

print("\nConfusion matrix saved.")

In [ ]:
# ============================================================
# Cell 8 — Train Final Model
# ============================================================

print("=" * 60)
print("Training Final Model")
print("=" * 60)

clf = xgb.XGBClassifier(**best_params)

clf.fit(
    X_scaled,
    y,
    sample_weight=sample_weights,
    verbose=False,
)

# ------------------------------------------------------------
# Save model
# ------------------------------------------------------------

MODEL_PATH = f"{OUTPUT_DIR}/xgb_4class_full.json"
SCALER_PATH = f"{OUTPUT_DIR}/scaler_4class.pkl"

clf.save_model(MODEL_PATH)
joblib.dump(scaler, SCALER_PATH)

print("\nModel successfully trained.\n")

print(f"Model  : {MODEL_PATH}")
print(f"Scaler : {SCALER_PATH}")

In [ ]:
# ============================================================
# Cell 9 — Feature Importance Analysis
# ============================================================

print("=" * 60)
print("Feature Importance")
print("=" * 60)

# ------------------------------------------------------------
# Compute importance
# ------------------------------------------------------------

importance = (
    pd.Series(
        clf.feature_importances_,
        index=FEATURE_COLS,
        name="importance"
    )
    .sort_values(ascending=False)
)

# ------------------------------------------------------------
# Print Top 20
# ------------------------------------------------------------

print("\nTop 20 Features\n")
print(importance.head(20).to_string())

print("\n")

zero_features = (importance == 0).sum()

print(f"Zero-importance features : {zero_features}/{len(FEATURE_COLS)}")

# ------------------------------------------------------------
# Save CSV
# ------------------------------------------------------------

importance.to_csv(
    f"{OUTPUT_DIR}/feature_importance.csv"
)

print("Feature importance CSV saved.")

# ------------------------------------------------------------
# Plot Top 20
# ------------------------------------------------------------

top20 = importance.head(20).sort_values()

plt.figure(figsize=(10,8))

plt.barh(
    top20.index,
    top20.values
)

plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Top 20 Most Important Features")

plt.tight_layout()

plt.savefig(
    f"{OUTPUT_DIR}/feature_importance_top20.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Feature importance plot saved.")

In [ ]:
# ============================================================
# Cell 10 — Summary
# ============================================================

print("="*60)
print("Training Complete")
print("="*60)

print(f"Dataset Size          : {len(df)}")
print(f"Feature Count         : {len(FEATURE_COLS)}")
print(f"Classes               : {len(CLASS_NAMES)}")
print(f"Best Macro F1         : {macro_f1:.4f}")
print(f"Balanced Accuracy     : {bal_acc:.4f}")

print("\nSaved Files")

print("----------------------------")
print("Model")
print("----------------------------")
print(MODEL_PATH)

print("\nScaler")
print("----------------------------")
print(SCALER_PATH)

print("\nFeature Importance")
print("----------------------------")
print(f"{OUTPUT_DIR}/feature_importance.csv")

print("\nFeature Importance Plot")
print("----------------------------")
print(f"{OUTPUT_DIR}/feature_importance_top20.png")

print("\nConfusion Matrix")
print("----------------------------")
print(f"{OUTPUT_DIR}/confusion_matrix_cv.png")

print("\nNotebook Finished Successfully.")